# Complete relation mapping run

This notebook creates one mapped relation CSV from Wikidata, EventKG, and WarSampo.

The code follows one repeated process:

1. Load the versioned relation map.
2. Reuse or download the three raw sources.
3. Convert each source assertion to the same small record shape.
4. Apply the common relation, direction, priority, and time rules.
5. Write one CSV and run data quality checks.

Set `RELATION_MAPPING_SAMPLE=1` for a small test output. Set
`RELATION_MAPPING_SKIP_DOWNLOADS=1` to use existing local raw data.


In [ ]:
# Install only missing packages. Set RELATION_MAPPING_SKIP_INSTALL=1 to skip this cell.
import importlib.util
import os
import subprocess
import sys

packages = {"requests": "requests", "pandas": "pandas", "rdflib": "rdflib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing and os.getenv("RELATION_MAPPING_SKIP_INSTALL", "0") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Package check complete. Missing:", missing)


In [ ]:
import csv
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable, Iterator
from urllib.parse import unquote

import pandas as pd
import requests
from IPython.display import display
from rdflib import Graph, Literal, OWL, URIRef


# Process 0: locate the repository so the notebook also works from its own folder.
def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "GG_Data" / "mapping").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the Corpus2GeoRAG_Pacific repository.")


ROOT = find_project_root()
DATA = ROOT / "Data" / "external"
WWII = DATA / "wwii_task_force"
EVENTKG = WWII / "eventkg_v3_2"
WARSAMPO = DATA / "warsampo"
OUTPUT_DIR = ROOT / "GG_Data" / "mapping" / "outputs" / "relation-map-v1.0.0"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_MODE = os.getenv("RELATION_MAPPING_SAMPLE", "0") == "1"
SAMPLE_LIMIT = 1_000 if SAMPLE_MODE else None
RUN_DOWNLOADS = os.getenv("RELATION_MAPPING_SKIP_DOWNLOADS", "0") != "1"
MAP_VERSION = "relation-map-v1.0.0"
MAP_URL = (
    "https://raw.githubusercontent.com/EtzionR/Corpus2GeoRAG_Pacific/"
    "main/GG_Data/mapping/RELATION_MAPPING_TABLE_v1.0.0.csv"
)
output_name = "complete_mapped_relations_sample.csv" if SAMPLE_MODE else "complete_mapped_relations_v1.0.0.csv"
FINAL_CSV = OUTPUT_DIR / output_name

print("Project root:", ROOT)
print("Mode:", "sample" if SAMPLE_MODE else "full")
print("Output:", FINAL_CSV)


## 1. Load the versioned relation map

The GitHub file is the first source. A local file is a fallback for development.
The run stops if the map has duplicate source predicates, missing required
fields, or the wrong version.


In [ ]:
# Process 1: load the versioned catalog that maps every source predicate.
REQUIRED_MAP_COLUMNS = {
    "source", "source_predicate", "final_relation_id", "final_relation_label",
    "direction_to_final", "final_mapping_class", "final_review_status",
    "level1_family", "level1_candidate", "level1_assignment_status",
    "selection_priority", "relation_map_version",
}
map_snapshot = OUTPUT_DIR / "RELATION_MAPPING_TABLE_v1.0.0.csv"
local_map = ROOT / "GG_Data" / "mapping" / "RELATION_MAPPING_TABLE_v1.0.0.csv"

try:
    response = requests.get(MAP_URL, timeout=60)
    response.raise_for_status()
    map_snapshot.write_bytes(response.content)
    map_origin = "GitHub"
except requests.RequestException as exc:
    if not local_map.exists():
        raise RuntimeError("The mapping table is not available from GitHub or the local repository.") from exc
    shutil.copy2(local_map, map_snapshot)
    map_origin = f"local fallback ({exc})"

relation_map_df = pd.read_csv(map_snapshot, dtype=str, keep_default_na=False)
missing = REQUIRED_MAP_COLUMNS - set(relation_map_df.columns)
assert not missing, f"Missing map columns: {sorted(missing)}"
assert set(relation_map_df["relation_map_version"]) == {MAP_VERSION}
assert not relation_map_df.duplicated(["source", "source_predicate"]).any()
assert set(relation_map_df["direction_to_final"]) <= {"same", "reverse"}
assert set(relation_map_df["selection_priority"]) <= {"1", "2", "3", "4"}

RELATION_MAP = {
    (row["source"], row["source_predicate"]): row
    for row in relation_map_df.to_dict("records")
}
print(f"Loaded {len(RELATION_MAP):,} mapping rows from {map_origin}.")
display(relation_map_df.groupby(["source", "selection_priority"]).size().rename("relation_types").reset_index())


## 2. Download or reuse raw data

- Wikipedia and Wikidata: the World War II task force population and its item JSON.
- EventKG 3.2: filtered, resumable scans linked to that Wikidata population.
- WarSampo 2.1.0: the complete Zenodo archive.

The cells use the repository download scripts because they already preserve
resume state, source records, and manifests.


In [ ]:
# Process 2: reuse complete local data. Download only a missing source.
EVENTKG_SECTIONS = {
    "events", "entities", "relations_events_base", "relations_events_other",
    "relations_entities_base", "relations_entities_other", "relations_entities_temporal",
}


def run_script(relative_path: str) -> None:
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Required repository script is missing: {path}")
    subprocess.run([sys.executable, "-u", str(path)], cwd=ROOT, check=True)


def safe_extract(archive: Path, target: Path) -> None:
    target_root = target.resolve()
    with zipfile.ZipFile(archive) as source_zip:
        for member in source_zip.infolist():
            destination = (target / member.filename).resolve()
            if destination != target_root and target_root not in destination.parents:
                raise RuntimeError(f"Unsafe archive path: {member.filename}")
        source_zip.extractall(target)


def file_md5(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "md5").hexdigest()


def download_warsampo() -> None:
    WARSAMPO.mkdir(parents=True, exist_ok=True)
    archive = WARSAMPO / "warsampo.zip"
    expected_md5 = "7bf298f712ae40675b144b86d8f50fc7"
    if not archive.exists() or file_md5(archive) != expected_md5:
        metadata_reply = requests.get("https://zenodo.org/api/records/3611322", timeout=60)
        metadata_reply.raise_for_status()
        entry = next(item for item in metadata_reply.json()["files"] if item["key"] == "warsampo.zip")
        temp = archive.with_suffix(".zip.part")
        with requests.get(entry["links"]["content"], stream=True, timeout=180) as reply:
            reply.raise_for_status()
            with temp.open("wb") as output:
                for chunk in reply.iter_content(1024 * 1024):
                    if chunk:
                        output.write(chunk)
        assert file_md5(temp) == expected_md5, "WarSampo checksum failed."
        temp.replace(archive)

    target = WARSAMPO / "full"
    if not target.exists() or not any(target.rglob("*.ttl")):
        target.mkdir(parents=True, exist_ok=True)
        safe_extract(archive, target)


if RUN_DOWNLOADS:
    if not (WWII / "manifest.json").exists() or not (WWII / "wikidata_entities.jsonl.gz").exists():
        run_script("scripts/fetch_wwii_task_force.py")
    incomplete = [name for name in EVENTKG_SECTIONS if not (EVENTKG / name / "manifest.json").exists()]
    if incomplete:
        print("Starting or resuming EventKG scans:", sorted(incomplete))
        run_script("scripts/run_wwii_task_force_full_scan.py")
    download_warsampo()

required_paths = [
    WWII / "article_qids.jsonl",
    WWII / "wikidata_entities.jsonl.gz",
    WARSAMPO / "full",
    *(EVENTKG / name / "manifest.json" for name in EVENTKG_SECTIONS),
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Raw data is incomplete:\n" + "\n".join(missing))
print("Raw source manifests are ready.")


## 3. Common mapping functions

The mapper changes direction only when the catalog says `reverse`. It keeps raw
time values and source evidence. A provisional catalog row stays provisional in
the output.


In [ ]:
# Process 3: apply one common mapping rule to records from every source.
OUTPUT_COLUMNS = [
    "observation_id", "subject_qid", "object_qid", "level1_relation",
    "level1_assignment_status", "level2_relation", "level2_label",
    "selection_priority", "selection_status", "direction", "time_from",
    "time_to", "time_point", "time_precision", "time_status",
    "selected_source", "selected_source_predicate", "source_record_id",
    "source_graph", "source_reference", "source_support", "evidence_references",
    "final_review_status", "final_mapping_class", "mapping_origin", "relation_map_version",
]
QID_RE = re.compile(r"^Q[1-9][0-9]*$")
WD_URI_RE = re.compile(r"https?://www\.wikidata\.org/entity/(Q[1-9][0-9]*)$")
SOURCE_PRIORITY = {"WarSampo": "2", "EventKG": "3", "Wikidata": "4"}

stats: Counter[str] = Counter()
top_relations: Counter[tuple[str, str]] = Counter()
runtime_predicates: set[tuple[str, str]] = set()


def compact_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def stable_id(*parts: str) -> str:
    payload = "\x1f".join(parts).encode("utf-8")
    return "obs-" + hashlib.sha256(payload).hexdigest()[:24]


def choose_one(values: Iterable[str]) -> tuple[str, list[str]]:
    unique = sorted({str(value) for value in values if value not in (None, "")})
    return (unique[0] if len(unique) == 1 else ""), unique


def parse_date(value: str) -> tuple[int, int, int] | None:
    match = re.match(r"^\+?(\d{4})-(\d{2})-(\d{2})", value)
    return tuple(map(int, match.groups())) if match else None


def relation_time_status(time_from: str, time_to: str, time_point: str, raw: dict[str, list[str]]) -> str:
    start, end = parse_date(time_from), parse_date(time_to)
    if start and end and start > end:
        return "CONFLICT"
    if any(len(values) > 1 for values in raw.values()):
        return "MULTIPLE"
    if time_point or (time_from and time_to):
        return "DIRECT"
    if time_from or time_to:
        return "PARTIAL"
    return "NONE"


def independent_mapping(source: str, predicate: str) -> dict[str, str]:
    # Keep an unseen predicate as its own Level 2 value. Do not guess equivalence.
    return {
        "direction_to_final": "same",
        "level1_family": "UNRESOLVED",
        "level1_candidate": "UNRESOLVED",
        "level1_assignment_status": "RUNTIME_UNRESOLVED",
        "final_relation_id": predicate,
        "final_relation_label": predicate.rsplit("/", 1)[-1],
        "selection_priority": SOURCE_PRIORITY[source],
        "final_review_status": "PROVISIONAL",
        "final_mapping_class": "INDEPENDENT_RUNTIME_UNRESOLVED",
        "relation_map_version": MAP_VERSION,
    }


def map_relation(
    *, source: str, predicate: str, subject_qid: str, object_qid: str,
    source_record_id: str, source_graph: str = "", source_reference: str = "",
    time_from: str = "", time_to: str = "", time_point: str = "",
    time_precision: dict[str, Any] | None = None,
    raw_time_values: dict[str, list[str]] | None = None,
) -> dict[str, str] | None:
    # Reject unresolved endpoints because the final graph uses Wikidata IDs only.
    if not QID_RE.fullmatch(subject_qid) or not QID_RE.fullmatch(object_qid):
        stats[f"{source}.invalid_qid"] += 1
        return None

    mapping = RELATION_MAP.get((source, predicate))
    mapping_origin = "catalog"
    if mapping is None:
        mapping = independent_mapping(source, predicate)
        mapping_origin = "runtime_independent"
        runtime_predicates.add((source, predicate))
        stats[f"{source}.runtime_independent_assertion"] += 1

    # Reverse endpoints only when the catalog defines the common direction as reverse.
    original_subject, original_object = subject_qid, object_qid
    direction = mapping["direction_to_final"]
    if direction == "reverse":
        subject_qid, object_qid = object_qid, subject_qid

    level1 = mapping["level1_family"]
    if level1 == "UNRESOLVED":
        level1 = mapping["level1_candidate"]

    raw_time_values = raw_time_values or {"from": [], "to": [], "point": []}
    observation_id = stable_id(
        source, source_record_id, source_graph, original_subject, predicate,
        original_object, compact_json(raw_time_values),
    )
    support = [{
        "source": source,
        "predicate": predicate,
        "record_id": source_record_id,
        "graph": source_graph,
        "subject_qid": original_subject,
        "object_qid": original_object,
    }]
    evidence = [{"reference": source_reference, "record_id": source_record_id, "graph": source_graph}]

    row = {
        "observation_id": observation_id,
        "subject_qid": subject_qid,
        "object_qid": object_qid,
        "level1_relation": level1,
        "level1_assignment_status": mapping["level1_assignment_status"],
        "level2_relation": mapping["final_relation_id"],
        "level2_label": mapping["final_relation_label"],
        "selection_priority": mapping["selection_priority"],
        "selection_status": "SELECTED",
        "direction": direction,
        "time_from": time_from,
        "time_to": time_to,
        "time_point": time_point,
        "time_precision": compact_json(time_precision or {}),
        "time_status": relation_time_status(time_from, time_to, time_point, raw_time_values),
        "selected_source": source,
        "selected_source_predicate": predicate,
        "source_record_id": source_record_id,
        "source_graph": source_graph,
        "source_reference": source_reference,
        "source_support": compact_json(support),
        "evidence_references": compact_json(evidence),
        "final_review_status": mapping["final_review_status"],
        "final_mapping_class": mapping["final_mapping_class"],
        "mapping_origin": mapping_origin,
        "relation_map_version": mapping["relation_map_version"],
    }
    stats[f"{source}.written"] += 1
    stats[f"priority.{mapping['selection_priority']}"] += 1
    stats[f"time.{row['time_status']}"] += 1
    top_relations[(source, mapping["final_relation_id"])] += 1
    return row


def write_source(writer: csv.DictWriter, source: str, records: Iterator[dict[str, Any]]) -> None:
    # All source readers end here, so writing and sample limits stay identical.
    written = 0
    for record in records:
        row = map_relation(source=source, **record)
        if row is None:
            continue
        writer.writerow(row)
        written += 1
        if SAMPLE_LIMIT and written >= SAMPLE_LIMIT:
            break
    print(f"{source}{' sample' if SAMPLE_MODE else ''}: {written:,} rows")


## 4. Prepare and map Wikidata assertions


In [ ]:
# Process 4: turn Wikidata item-valued statements into standard source records.
def wikidata_time(statement: dict[str, Any]) -> tuple[str, str, str, dict[str, Any], dict[str, list[str]]]:
    qualifiers = statement.get("qualifiers", {})

    def values(prop: str) -> tuple[str, list[str], list[int]]:
        found: list[str] = []
        precisions: list[int] = []
        for snak in qualifiers.get(prop, []):
            value = snak.get("datavalue", {}).get("value")
            if isinstance(value, dict) and value.get("time"):
                found.append(value["time"])
                if value.get("precision") is not None:
                    precisions.append(int(value["precision"]))
        selected, raw = choose_one(found)
        return selected, raw, sorted(set(precisions))

    time_from, raw_from, precision_from = values("P580")
    time_to, raw_to, precision_to = values("P582")
    time_point, raw_point, precision_point = values("P585")
    precision = {"from": precision_from, "to": precision_to, "point": precision_point}
    raw = {"from": raw_from, "to": raw_to, "point": raw_point}
    return time_from, time_to, time_point, precision, raw


def wikidata_records() -> Iterator[dict[str, Any]]:
    with gzip.open(WWII / "wikidata_entities.jsonl.gz", "rt", encoding="utf-8") as stream:
        for line in stream:
            record = json.loads(line)
            entity = record.get("entity") or {}
            subject = entity.get("id") or record.get("requested_qid", "")
            if not QID_RE.fullmatch(subject):
                stats["Wikidata.missing_entity"] += 1
                continue

            for prop, statements in entity.get("claims", {}).items():
                for statement in statements:
                    if statement.get("rank") == "deprecated":
                        stats["Wikidata.deprecated"] += 1
                        continue
                    value = statement.get("mainsnak", {}).get("datavalue", {}).get("value")
                    if not isinstance(value, dict) or value.get("entity-type") != "item":
                        continue

                    target = value.get("id") or (f"Q{value['numeric-id']}" if value.get("numeric-id") else "")
                    predicate = f"wd:{prop}"
                    time_from, time_to, time_point, precision, raw = wikidata_time(statement)
                    statement_id = statement.get("id") or stable_id(subject, predicate, target, compact_json(raw))
                    references = [ref.get("hash", "") for ref in statement.get("references", [])]
                    yield {
                        "predicate": predicate,
                        "subject_qid": subject,
                        "object_qid": target,
                        "source_record_id": statement_id,
                        "source_graph": "https://www.wikidata.org",
                        "source_reference": compact_json({
                            "statement_id": statement_id,
                            "reference_hashes": references,
                        }),
                        "time_from": time_from,
                        "time_to": time_to,
                        "time_point": time_point,
                        "time_precision": precision,
                        "raw_time_values": raw,
                    }


## 5. Prepare and map EventKG assertions


In [ ]:
# Process 5: read EventKG's direct triples and reified relation records.
URI_QUAD = re.compile(r"^<([^>]+)> <([^>]+)> <([^>]+)> (?:<([^>]+)> )?\.$")
LITERAL_QUAD = re.compile(
    r'^<([^>]+)> <([^>]+)> "((?:\\.|[^"\\])*)"(?:\^\^<[^>]+>|@[A-Za-z0-9-]+)? (?:<([^>]+)> )?\.$'
)
RDF_SUBJECT = "http://www.w3.org/1999/02/22-rdf-syntax-ns#subject"
RDF_OBJECT = "http://www.w3.org/1999/02/22-rdf-syntax-ns#object"
SEM_ROLE = "http://semanticweb.cs.vu.nl/2009/11/sem/roleType"
SEM_BEGIN = "http://semanticweb.cs.vu.nl/2009/11/sem/hasBeginTimeStamp"
SEM_END = "http://semanticweb.cs.vu.nl/2009/11/sem/hasEndTimeStamp"
START_UNIT = "https://eventkg.l3s.uni-hannover.de/schema/startUnitType"
END_UNIT = "https://eventkg.l3s.uni-hannover.de/schema/endUnitType"
STRUCTURAL_PREDICATES = {
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "http://www.w3.org/2002/07/owl#sameAs",
}
DIRECT_EVENTKG_SECTIONS = ("events", "entities", "relations_events_base", "relations_entities_base")
REIFIED_EVENTKG_SECTIONS = ("relations_events_other", "relations_entities_temporal", "relations_entities_other")


def part_paths(directory: Path) -> list[Path]:
    paths = sorted(directory.glob("part_*.nq.gz"))
    return paths[:2] if SAMPLE_MODE else paths


def eventkg_qids() -> tuple[dict[str, str], set[str]]:
    candidates: dict[str, set[str]] = defaultdict(set)
    for name in ("events", "entities"):
        paths = sorted((EVENTKG / name).glob("part_*.qid_map.tsv"))
        for path in paths[:20] if SAMPLE_MODE else paths:
            with path.open(encoding="utf-8") as stream:
                for line in stream:
                    qid, resource = line.rstrip("\n").split("\t", 1)
                    candidates[resource.strip("<>")].add(qid)
    ambiguous = {resource for resource, qids in candidates.items() if len(qids) != 1}
    resolved = {resource: next(iter(qids)) for resource, qids in candidates.items() if len(qids) == 1}
    return resolved, ambiguous


def qid_for_uri(uri: str, resource_qids: dict[str, str]) -> str:
    if uri in resource_qids:
        return resource_qids[uri]
    match = WD_URI_RE.fullmatch(uri)
    return match.group(1) if match else ""


def iter_nquad_groups(directory: Path) -> Iterator[tuple[str, list[str]]]:
    # EventKG sorts relation files by subject. Yield one subject block at a time.
    for path in part_paths(directory):
        current_subject = ""
        lines: list[str] = []
        with gzip.open(path, "rt", encoding="utf-8") as stream:
            for line in stream:
                subject = line.split(" ", 1)[0]
                if current_subject and subject != current_subject:
                    yield path.name, lines
                    lines = []
                current_subject = subject
                lines.append(line.rstrip("\n"))
        if lines:
            yield path.name, lines


def eventkg_event_dates() -> dict[tuple[str, str], dict[str, list[str]]]:
    dates: dict[tuple[str, str], dict[str, set[str]]] = defaultdict(lambda: defaultdict(set))
    for _, lines in iter_nquad_groups(EVENTKG / "relations_events_base"):
        for line in lines:
            literal = LITERAL_QUAD.match(line)
            uri = URI_QUAD.match(line)
            if literal:
                resource, predicate, value, graph = literal.groups()
                key = (resource, graph or "source_unknown")
                if predicate == SEM_BEGIN:
                    dates[key]["from"].add(value)
                elif predicate == SEM_END:
                    dates[key]["to"].add(value)
            elif uri:
                resource, predicate, value, graph = uri.groups()
                key = (resource, graph or "source_unknown")
                if predicate == START_UNIT:
                    dates[key]["from_precision"].add(value.rsplit("#", 1)[-1])
                elif predicate == END_UNIT:
                    dates[key]["to_precision"].add(value.rsplit("#", 1)[-1])
    return {key: {field: sorted(values) for field, values in fields.items()} for key, fields in dates.items()}


def direct_eventkg_records(
    resource_qids: dict[str, str],
    dates: dict[tuple[str, str], dict[str, list[str]]],
) -> Iterator[dict[str, Any]]:
    for name in DIRECT_EVENTKG_SECTIONS:
        for part, lines in iter_nquad_groups(EVENTKG / name):
            for line_number, line in enumerate(lines, start=1):
                match = URI_QUAD.match(line)
                if not match:
                    continue
                subject_uri, predicate, object_uri, graph = match.groups()
                subject = qid_for_uri(subject_uri, resource_qids)
                target = qid_for_uri(object_uri, resource_qids)
                if not subject or not target or predicate in STRUCTURAL_PREDICATES:
                    continue

                raw = dates.get((subject_uri, graph or "source_unknown"), {})
                time_from, raw_from = choose_one(raw.get("from", []))
                time_to, raw_to = choose_one(raw.get("to", []))
                yield {
                    "predicate": predicate,
                    "subject_qid": subject,
                    "object_qid": target,
                    "source_record_id": stable_id(name, part, str(line_number), line),
                    "source_graph": graph or "source_unknown",
                    "source_reference": f"{name}/{part}:{line_number}",
                    "time_from": time_from,
                    "time_to": time_to,
                    "time_precision": {
                        "from": raw.get("from_precision", []),
                        "to": raw.get("to_precision", []),
                    },
                    "raw_time_values": {"from": raw_from, "to": raw_to, "point": []},
                }


def reified_eventkg_records(resource_qids: dict[str, str]) -> Iterator[dict[str, Any]]:
    for name in REIFIED_EVENTKG_SECTIONS:
        for part, lines in iter_nquad_groups(EVENTKG / name):
            relation_uri = lines[0].split(" ", 1)[0].strip("<>")
            by_graph: dict[str, dict[str, list[str]]] = defaultdict(lambda: defaultdict(list))
            for line in lines:
                match = URI_QUAD.match(line) or LITERAL_QUAD.match(line)
                if match:
                    _, predicate, value, graph = match.groups()
                    by_graph[graph or "source_unknown"][predicate].append(value)

            for graph, values in by_graph.items():
                if not all(values.get(key) for key in (RDF_SUBJECT, RDF_OBJECT, SEM_ROLE)):
                    continue
                subject_uri, object_uri = values[RDF_SUBJECT][0], values[RDF_OBJECT][0]
                subject = qid_for_uri(subject_uri, resource_qids)
                target = qid_for_uri(object_uri, resource_qids)
                if not subject or not target:
                    stats["EventKG.unresolved_endpoint"] += 1
                    continue

                time_from, raw_from = choose_one(values.get(SEM_BEGIN, []))
                time_to, raw_to = choose_one(values.get(SEM_END, []))
                yield {
                    "predicate": values[SEM_ROLE][0],
                    "subject_qid": subject,
                    "object_qid": target,
                    "source_record_id": f"{relation_uri}|{graph}",
                    "source_graph": graph,
                    "source_reference": f"{name}/{part}:{relation_uri}",
                    "time_from": time_from,
                    "time_to": time_to,
                    "time_precision": {
                        "from": sorted({item.rsplit("#", 1)[-1] for item in values.get(START_UNIT, [])}),
                        "to": sorted({item.rsplit("#", 1)[-1] for item in values.get(END_UNIT, [])}),
                    },
                    "raw_time_values": {"from": raw_from, "to": raw_to, "point": []},
                }


def eventkg_records() -> Iterator[dict[str, Any]]:
    resource_qids, ambiguous = eventkg_qids()
    stats["EventKG.ambiguous_identity_resources"] = len(ambiguous)
    yield from direct_eventkg_records(resource_qids, eventkg_event_dates())
    yield from reified_eventkg_records(resource_qids)


## 6. Prepare and map the complete WarSampo data


In [ ]:
# Process 6: resolve WarSampo resources to QIDs, then emit mapped URI relations.
def load_article_qids() -> dict[str, str]:
    article_qids: dict[str, str] = {}
    with (WWII / "article_qids.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line)
            if not row.get("qid"):
                continue
            for field in ("article_title", "resolved_title"):
                if row.get(field):
                    article_qids[row[field].replace(" ", "_")] = row["qid"]
    return article_qids


def has_identity_link(path: Path) -> bool:
    content = path.read_bytes()
    return b"wikipedia.org" in content or b"wikidata.org" in content


def warsampo_identity_map(ttl_paths: list[Path]) -> tuple[dict[str, str], set[str]]:
    article_qids = load_article_qids()
    candidates: dict[str, set[str]] = defaultdict(set)
    rejected_pairs: set[tuple[str, str]] = set()

    # Keep the earlier manual review decisions when they are available.
    audit_path = ROOT / "poc" / "relation_alignment" / "audit_2026-10-11" / "identity_candidates.csv"
    if audit_path.exists():
        with audit_path.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                pair = (row["warsampo_uri"], row["wikidata_qid"])
                if row["review_status"] == "rejected":
                    rejected_pairs.add(pair)
                elif row["review_status"] == "reviewed":
                    candidates[pair[0]].add(pair[1])

    for path in (path for path in ttl_paths if has_identity_link(path)):
        graph = Graph()
        graph.parse(path, format="turtle")
        for subject, _, target in graph.triples((None, OWL.sameAs, None)):
            target_text = str(target)
            match = WD_URI_RE.fullmatch(target_text)
            if match:
                pair = (str(subject), match.group(1))
                if pair not in rejected_pairs:
                    candidates[pair[0]].add(pair[1])
                continue
            marker = "en.wikipedia.org/wiki/"
            if marker in target_text:
                title = unquote(target_text.split(marker, 1)[1]).replace(" ", "_")
                if title in article_qids:
                    candidates[str(subject)].add(article_qids[title])

    ambiguous = {resource for resource, qids in candidates.items() if len(qids) != 1}
    resolved = {resource: next(iter(qids)) for resource, qids in candidates.items() if len(qids) == 1}
    return resolved, ambiguous


def local_name(uri: str) -> str:
    return uri.rsplit("/", 1)[-1].rsplit("#", 1)[-1]


def literal_precision(value: str) -> str:
    for pattern, precision in (
        (r"[+-]?\d{4}", "year"),
        (r"[+-]?\d{4}-\d{2}", "month"),
        (r"[+-]?\d{4}-\d{2}-\d{2}", "day"),
    ):
        if re.fullmatch(pattern, value):
            return precision
    return "source"


def warsampo_records() -> Iterator[dict[str, Any]]:
    ttl_paths = sorted((WARSAMPO / "full").rglob("*.ttl"))
    resource_qids, ambiguous = warsampo_identity_map(ttl_paths)
    stats["WarSampo.ambiguous_identity_resources"] = len(ambiguous)

    scan_paths = ttl_paths
    if SAMPLE_MODE:
        identity_files = [path for path in ttl_paths if "actor_data_persons" in path.name or "unit_data" in path.name]
        scan_paths = list(dict.fromkeys(identity_files + ttl_paths[:5]))

    # First collect time spans and the small set of relations with two resolved QIDs.
    relations: list[dict[str, str]] = []
    subject_spans: dict[str, set[str]] = defaultdict(set)
    span_from: dict[str, set[str]] = defaultdict(set)
    span_to: dict[str, set[str]] = defaultdict(set)
    for path in scan_paths:
        graph = Graph()
        graph.parse(path, format="turtle")
        relative = path.relative_to(WARSAMPO / "full").as_posix()
        for subject, predicate_node, obj in graph:
            predicate = str(predicate_node)
            name = local_name(predicate)
            if name == "P4_has_time-span" and isinstance(obj, URIRef):
                subject_spans[str(subject)].add(str(obj))
            elif name == "P82a_begin_of_the_begin" and isinstance(obj, Literal):
                span_from[str(subject)].add(str(obj))
            elif name == "P82b_end_of_the_end" and isinstance(obj, Literal):
                span_to[str(subject)].add(str(obj))

            if not isinstance(subject, URIRef) or not isinstance(obj, URIRef):
                continue
            subject_qid = resource_qids.get(str(subject), "")
            object_qid = resource_qids.get(str(obj), "")
            if not subject_qid or not object_qid or predicate in STRUCTURAL_PREDICATES:
                continue
            if ("WarSampo", predicate) not in RELATION_MAP:
                stats["WarSampo.unmapped_predicate"] += 1
                continue
            relations.append({
                "subject_uri": str(subject),
                "object_uri": str(obj),
                "predicate": predicate,
                "file": relative,
                "record_id": stable_id(relative, str(subject), predicate, str(obj)),
            })

    # Join each relation subject to its CRM time span after all TTL files are read.
    for relation in relations:
        subject_uri = relation["subject_uri"]
        from_values = sorted({
            value for span in subject_spans.get(subject_uri, set()) for value in span_from.get(span, set())
        })
        to_values = sorted({
            value for span in subject_spans.get(subject_uri, set()) for value in span_to.get(span, set())
        })
        time_from, raw_from = choose_one(from_values)
        time_to, raw_to = choose_one(to_values)
        yield {
            "predicate": relation["predicate"],
            "subject_qid": resource_qids[subject_uri],
            "object_qid": resource_qids[relation["object_uri"]],
            "source_record_id": relation["record_id"],
            "source_graph": "WarSampo 2.1.0",
            "source_reference": relation["file"],
            "time_from": time_from,
            "time_to": time_to,
            "time_precision": {
                "from": sorted({literal_precision(value) for value in raw_from}),
                "to": sorted({literal_precision(value) for value in raw_to}),
            },
            "raw_time_values": {"from": raw_from, "to": raw_to, "point": []},
        }


## 7. Run the mapping

The output is first written to a temporary file. It replaces the final CSV only
after all three source mappers finish.


In [ ]:
# Process 7: stream all three sources into one atomic CSV output.
stats.clear()
top_relations.clear()
runtime_predicates.clear()

temp_output = FINAL_CSV.with_suffix(".csv.part")
with temp_output.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    for source_name, records in (
        ("Wikidata", wikidata_records()),
        ("EventKG", eventkg_records()),
        ("WarSampo", warsampo_records()),
    ):
        write_source(writer, source_name, records)

# Replace the final file only after every source finishes successfully.
temp_output.replace(FINAL_CSV)
print("Saved:", FINAL_CSV)
print("Bytes:", f"{FINAL_CSV.stat().st_size:,}")


## 8. EDA and data quality tests


In [ ]:
# Process 8: read the final CSV once for EDA and required data quality checks.
dq = Counter()
breakdowns = {
    "selected_source": Counter(),
    "selection_priority": Counter(),
    "final_review_status": Counter(),
    "level1_relation": Counter(),
    "time_status": Counter(),
    "mapping_origin": Counter(),
}
seen_ids: set[str] = set()
examples: list[dict[str, str]] = []

with FINAL_CSV.open(encoding="utf-8", newline="") as stream:
    reader = csv.DictReader(stream)
    dq["schema_matches"] = int(reader.fieldnames == OUTPUT_COLUMNS)
    for row in reader:
        dq["rows"] += 1
        if len(examples) < 5:
            examples.append(row)
        if row["observation_id"] in seen_ids:
            dq["duplicate_observation_id"] += 1
        seen_ids.add(row["observation_id"])

        dq["invalid_qid"] += int(
            not QID_RE.fullmatch(row["subject_qid"]) or not QID_RE.fullmatch(row["object_qid"])
        )
        dq["missing_relation"] += int(not row["level1_relation"] or not row["level2_relation"])
        dq["invalid_priority"] += int(row["selection_priority"] not in {"1", "2", "3", "4"})
        dq["invalid_direction"] += int(row["direction"] not in {"same", "reverse"})
        dq["wrong_map_version"] += int(row["relation_map_version"] != MAP_VERSION)
        dq["invalid_mapping_origin"] += int(row["mapping_origin"] not in {"catalog", "runtime_independent"})

        start, end = parse_date(row["time_from"]), parse_date(row["time_to"])
        if start and end and start > end:
            if row["time_status"] == "CONFLICT":
                dq["flagged_invalid_period"] += 1
            else:
                dq["unflagged_invalid_period"] += 1

        for field, counter in breakdowns.items():
            counter[row[field]] += 1

checks = [
    ("Output has rows", dq["rows"] > 0, dq["rows"]),
    ("Schema matches", dq["schema_matches"] == 1, dq["schema_matches"]),
    ("Observation IDs are unique", dq["duplicate_observation_id"] == 0, dq["duplicate_observation_id"]),
    ("All endpoints are Wikidata QIDs", dq["invalid_qid"] == 0, dq["invalid_qid"]),
    ("All rows have Level 1 and Level 2", dq["missing_relation"] == 0, dq["missing_relation"]),
    ("All priorities are valid", dq["invalid_priority"] == 0, dq["invalid_priority"]),
    ("All directions are valid", dq["invalid_direction"] == 0, dq["invalid_direction"]),
    ("All rows use the map version", dq["wrong_map_version"] == 0, dq["wrong_map_version"]),
    ("Invalid periods are marked CONFLICT", dq["unflagged_invalid_period"] == 0, dq["unflagged_invalid_period"]),
    ("Every row states its mapping origin", dq["invalid_mapping_origin"] == 0, dq["invalid_mapping_origin"]),
]
dq_df = pd.DataFrame(checks, columns=["test", "passed", "failures"])
display(dq_df)
assert dq_df["passed"].all(), "One or more data quality tests failed."

# Show the main EDA tables after the quality gate passes.
for field in ("selected_source", "selection_priority", "final_review_status", "time_status", "mapping_origin"):
    display(pd.DataFrame(breakdowns[field].most_common(), columns=[field, "rows"]))
if runtime_predicates:
    display(pd.DataFrame(sorted(runtime_predicates), columns=["source", "unseen_predicate"]))
display(pd.DataFrame(breakdowns["level1_relation"].most_common(20), columns=["level1_relation", "rows"]))
display(pd.DataFrame([
    {key: row[key] for key in (
        "observation_id", "subject_qid", "level2_relation",
        "object_qid", "selected_source", "time_status",
    )}
    for row in examples
]))
print(f"DQ passed. Final CSV: {FINAL_CSV}")
